# Camada Ouro: Dados prontos para o uso

Joins distribuídos e agregações em larga escala.

# Configuração do Ambiente

In [0]:
########################################################
#### LIBS
########################################################

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# Leitura dos dados

In [0]:
########################################################
#### LEITURA DOS DADOS - CÓDIGO MUNICÍPIOS IBGE
########################################################
ibge = pd.read_excel("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/RELATORIO_DTB_BRASIL_MUNICIPIO.xls")

ibge = spark.createDataFrame(ibge)
ibge = ibge.select(["Nome_UF", "Código Município Completo", "Nome_Município"])
ibge.show(5)

+--------+-------------------------+--------------------+
| Nome_UF|Código Município Completo|      Nome_Município|
+--------+-------------------------+--------------------+
|Rondônia|                  1100015|Alta Floresta D'O...|
|Rondônia|                  1100379|Alto Alegre dos P...|
|Rondônia|                  1100403|        Alto Paraíso|
|Rondônia|                  1100346|    Alvorada D'Oeste|
|Rondônia|                  1100023|           Ariquemes|
+--------+-------------------------+--------------------+
only showing top 5 rows


In [0]:
municipio.select("rede").distinct().show()

+----+
|rede|
+----+
|   2|
|   3|
|   5|
|   0|
+----+



In [0]:
########################################################
#### LEITURA DOS DADOS 
########################################################
#
streaming = spark.read.format("parquet").load("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/streaming/")
streaming = streaming.select(["id_municipio", "ano", "rede", "serie", "taxa_alfabetizacao", "media_portugues"])

streaming.show(10)

#
municipio = spark.read.format("parquet").load("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/municipio/")
municipio = municipio.select(["id_municipio", "ano", "rede", "serie", "taxa_alfabetizacao", "media_portugues"])

municipio.show(10)



+------------+----+----+-----+------------------+---------------+
|id_municipio| ano|rede|serie|taxa_alfabetizacao|media_portugues|
+------------+----+----+-----+------------------+---------------+
|     3509502|2025|   4|    2|              62.3|          524.7|
|     3509502|2025|   3|    2|              49.8|          644.6|
|     3509502|2025|   3|    2|              64.3|          918.6|
|     3550308|2025|   3|    2|              75.0|          324.7|
|     3509502|2025|   3|    2|              63.0|          806.7|
|     3550308|2025|   4|    2|              90.5|          546.8|
|     3550308|2025|   3|    2|              51.4|          830.4|
|     3550308|2025|   3|    2|              52.6|          340.4|
|     3518800|2025|   2|    2|              81.4|          555.0|
|     3518800|2025|   4|    2|              53.8|          411.3|
+------------+----+----+-----+------------------+---------------+
only showing top 10 rows
+------------+----+----+-----+------------------+--

# Agregação e join dos dados de batch + streaming

In [0]:
########################################################
#### UNINDO OS DOIS TIPOS DE DADOS
########################################################
municipio_unido = municipio.unionByName(streaming)
municipio_unido.show(10)

+------------+----+----+-----+------------------+---------------+
|id_municipio| ano|rede|serie|taxa_alfabetizacao|media_portugues|
+------------+----+----+-----+------------------+---------------+
|     3557154|2024|   5|    2|              80.0|       784.6376|
|     3547650|2024|   2|    2|             95.45|       811.8048|
|     3146909|2024|   2|    2|             100.0|         783.99|
|     2201988|2024|   3|    2|             95.35|         796.92|
|     2201988|2024|   5|    2|             95.35|         796.92|
|     3117603|2024|   3|    2|             100.0|          793.7|
|     3553955|2024|   3|    2|             91.72|       795.9251|
|     2304269|2024|   5|    2|             98.82|         817.13|
|     3514700|2024|   3|    2|             95.07|       791.2692|
|     5106281|2024|   3|    2|             94.16|         798.14|
+------------+----+----+-----+------------------+---------------+
only showing top 10 rows


In [0]:
########################################################
#### JOIN E RENOMEANDO OS DADOS
########################################################
#
municipio_unido = municipio_unido.join(
    ibge.select("Código Município Completo", "Nome_Município").withColumnRenamed("Código Município Completo", "id_municipio"),
    on="id_municipio",
    how="left"
)
#
municipio_unido = municipio_unido.withColumnsRenamed({
    "id_municipio": "ID_MUNICIPIO",
    "ano": "ANO",
    "rede": "REDE",
    "serie": "SERIE",
    "taxa_alfabetizacao": " TAXA_ALFABETIZACAO",
    "media_portugues": "MEDIA_PORTUGUES_SAEB",
    "Nome_Município": "NOME_MUNICIPIO"})
#
municipio_unido.show(5)

+------------+----+----+-----+-------------------+--------------------+--------------+
|ID_MUNICIPIO| ANO|REDE|SERIE| TAXA_ALFABETIZACAO|MEDIA_PORTUGUES_SAEB|NOME_MUNICIPIO|
+------------+----+----+-----+-------------------+--------------------+--------------+
|     3557154|2024|   5|    2|               80.0|            784.6376|      Zacarias|
|     3547650|2024|   2|    2|              95.45|            811.8048|  Santa Salete|
|     3146909|2024|   2|    2|              100.0|              783.99|     Papagaios|
|     2201988|2024|   3|    2|              95.35|              796.92|Brejo do Piauí|
|     2201988|2024|   5|    2|              95.35|              796.92|Brejo do Piauí|
+------------+----+----+-----+-------------------+--------------------+--------------+
only showing top 5 rows


In [0]:
########################################################
#### CHECAGEM
########################################################

print(f"Número de linhas: {municipio_unido.count()}")

Número de linhas: 26731


# Exportação

In [0]:
municipio_unido.write.partitionBy("ano").mode("overwrite").parquet("/Volumes/workspace/default/inep_avaliacao_alfabetizacao/gold/municipio_unido")